# Step 3 — A ReAct Agent From Scratch

**Agent = LLM + tools + a loop.** The LLM decides *which* tool to use; **our code** runs it and feeds the result back as an *Observation*. The loop repeats until the model writes `Final Answer:`.

```
Thought:      the model reasons about what to do
Action:       it names a tool
Action Input: and the input for it
Observation:  OUR code runs the tool and shows the result
... repeat ...
Final Answer: the answer for the user
```

### Setup
Run this cell first. It lets the notebook import the lab's own code (`config.py`, `rag/`, `agent/`, `llm.py`) from the folder above `notebooks/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

In [ ]:
from rag.ingest import build_index

build_index()  # the search_documents tool needs the index

## 1. The tools are just Python functions
The LLM never runs code. It only *asks* for a tool by name. Let's call the tools ourselves first.

In [ ]:
from agent.tools import TOOL_REGISTRY, calculator, search_documents

for name, entry in TOOL_REGISTRY.items():
    print(f"- {name}: {entry['description'][:90]}...")

In [ ]:
print(calculator("18/100*4520 + 75"))
print(calculator("__import__('os')"))   # unsafe input is refused: the calculator never uses eval()

In [ ]:
print(search_documents("Sick Leave carry forward")[:600])

## 2. The system prompt teaches the model the format
The tool list is pasted in as plain text, together with the exact reply format.

In [ ]:
from agent.react_agent import build_system_prompt

print(build_system_prompt())

## 3. Reading the model's reply
`parse_reply()` uses regular expressions to pull out the parts. Here it is on a hand-written reply, no LLM needed.

In [ ]:
from agent.react_agent import parse_reply

sample = '''Thought: This is a policy question, I should search.
Action: search_documents
Action Input: Sick Leave carry forward'''
parse_reply(sample)

## 4. Run the agent
Each question prints its full trace. Watch which tool it picks.

In [ ]:
from llm import check_llm

ok, message = check_llm()
print("LLM ready ✅" if ok else f"LLM NOT ready ❌  {message}")

In [ ]:
DEMO_QUESTIONS = [
    ("What is 18% of 4520, plus 75?", "calculator"),
    ("How many days of Sick Leave can be carried forward?", "search_documents"),
    ("What is the Maternity Benefit Act in India?", "wikipedia_lookup"),
    ("I joined in March. How many Earned Leave days will I have accrued by the end of September, "
     "and can I encash them?", "search_documents, then calculator (7 x 1.5 = 10.5)"),
]

In [ ]:
from agent import print_trace
from agent.react_agent import run_react_agent

for question, expected in DEMO_QUESTIONS:
    print("=" * 80)
    print("Q:", question, f"   (expected: {expected})")
    print_trace(run_react_agent(question))

## What to notice
- The calculator question never touches the documents; the policy question never touches Wikipedia.
- The last question needs **two** tools in a row. A small 3B model often gets this one wrong; a larger model solves it.

✏️ **Try it:** write your own question below.

In [ ]:
print_trace(run_react_agent("How many Casual Leave days do I get, and what is that per quarter?"))